# Prompt-to-Prompt Image Editing with Cross-Attention Control
**Author:** Anuj Yadav (Indian Institute of Technology Kharagpur)

Based on the foundational research: *Prompt-to-Prompt Image Editing with Cross Attention Control* (Hertz et al., Google Research, 2022).

This notebook demonstrates:
1. Model loading and device detection (Apple Silicon MPS / CUDA / CPU)
2. Word Swap editing ("A dog sitting on a beach" → "A cat sitting on a beach")
3. Prompt Refinement ("A dog in a park" → "A small golden dog in a park")
4. Attention Re-weighting (Equalizer)
5. Cross-attention map extraction and spatial heatmap visualization
6. Quantitative structural fidelity and edit evaluation

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
from PIL import Image
from src.models.loader import load_model
from src.pipeline.editing import edit_word_swap, edit_prompt_refinement, edit_reweight
from src.attention.visualization import get_token_attention_map, create_attention_overlay
from src.evaluation.comparison import generate_evaluation_report
from src.utils.device import get_device_info

print("Device diagnostics:", get_device_info())
pipe = load_model()

## 1. Experiment 1: Word Swap
We substitute the token "dog" with "cat" while retaining the exact beach background, composition, and animal posture.

In [ ]:
src_prompt = "A photo of a dog sitting on a beach"
tgt_prompt = "A photo of a cat sitting on a beach"

img_orig, img_edit, controller, alignment = edit_word_swap(
    pipe=pipe,
    source_prompt=src_prompt,
    target_prompt=tgt_prompt,
    cross_replace_steps=0.8,
    self_replace_steps=0.4,
    num_inference_steps=25,
    seed=42,
)

from src.utils.image import create_side_by_side
comparison = create_side_by_side(img_orig, img_edit, label1="Original (Dog)", label2="Edited (Cat)")
display(comparison)

## 2. Visualizing Cross-Attention Heatmaps

In [ ]:
if alignment.get('changed_tokens'):
    src_idx = alignment['changed_tokens'][0]['src_idx']
    tgt_idx = alignment['changed_tokens'][0]['tgt_idx']
    
    heat_src = get_token_attention_map(controller, src_idx, batch_idx=0)
    overlay = create_attention_overlay(img_orig, heat_src)
    display(overlay)

## 3. Quantitative Evaluation Report

In [ ]:
report = generate_evaluation_report(img_orig, img_edit, src_prompt, tgt_prompt, pipe=pipe)
print("Evaluation Report:")
for k, v in report['structural_fidelity'].items():
    print(f"  {k}: {v}")